# PubMedQA EDA

Exploratory analysis for `qiaojin/PubMedQA` configs `pqa_artificial`, `pqa_labeled`, and `pqa_unlabeled`. The goal is to compare label coverage, context length, and answer distributions before building CoCA-ready examples.

In [ ]:
from datasets import load_dataset
import pandas as pd

configs = ["pqa_artificial", "pqa_labeled", "pqa_unlabeled"]
datasets = {config: load_dataset("qiaojin/PubMedQA", config, split="train") for config in configs}
{config: len(split) for config, split in datasets.items()}

In [ ]:
frames = []
for config, split_ds in datasets.items():
    frame = split_ds.to_pandas()
    frame["config"] = config
    frame["question_chars"] = frame["question"].str.len()
    frame["question_words"] = frame["question"].str.split().str.len()
    frame["context_count"] = frame["context"].apply(lambda c: len(c.get("contexts", [])) if isinstance(c, dict) else 0)
    frame["context_chars"] = frame["context"].apply(lambda c: sum(len(x) for x in c.get("contexts", [])) if isinstance(c, dict) else 0)
    frames.append(frame)

pubmedqa = pd.concat(frames, ignore_index=True)
pubmedqa.head()

In [ ]:
summary = {
    "config_sizes": {config: len(split_ds) for config, split_ds in datasets.items()},
    "decision_distribution": pubmedqa.groupby(["config", "final_decision"]).size().unstack(fill_value=0),
    "lengths_by_config": pubmedqa.groupby("config")[["question_words", "context_count", "context_chars"]].describe(),
    "missing_decisions": pubmedqa["final_decision"].isna().groupby(pubmedqa["config"]).sum(),
}

summary